# Attention as Token-Measure Flow

This notebook generates `fig:transformer-token-measure-flow`.  Tokens are represented as an empirical measure, and an attention layer transports each token by a context-dependent weighted average.  We use the attractive setup $Q=K=I$ and velocity $W(X)X-X$, so repeated residual layers cluster nearby tokens.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb

NAME = "transformer-token-measure-flow"
out = figure_dir(NAME)
rng = np.random.default_rng(45)

The example is schematic, not a trained transformer.  The coordinates are rescaled mildly after each layer to keep the token flow readable while preserving the clustering pattern.

In [2]:
X0 = np.vstack([
    rng.normal(loc=[-0.88, -0.20], scale=[0.18, 0.22], size=(5, 2)),
    rng.normal(loc=[0.10, 0.76], scale=[0.20, 0.16], size=(4, 2)),
    rng.normal(loc=[0.82, -0.26], scale=[0.17, 0.20], size=(5, 2)),
])
tau = 0.72
temp = 0.28
layers = [X0.copy()]
weights = []
X = X0.copy()
for _ in range(5):
    logits = X @ X.T
    logits = logits - logits.max(axis=1, keepdims=True)
    W = np.exp(logits / temp)
    W = W / W.sum(axis=1, keepdims=True)
    weights.append(W)
    V = W @ X - X
    X = X + tau * V
    X = X - X.mean(axis=0, keepdims=True) + layers[0].mean(axis=0, keepdims=True)
    layers.append(X.copy())
all_points = np.vstack(layers)
xlim, ylim = padded_limits(all_points, pad=0.18)

In [3]:
def decorate(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    remove_axes(ax)

fig, ax = plt.subplots(figsize=(2.35, 2.05))
W = weights[0]
segments=[]; colors=[]; widths=[]
base_violet = np.array(to_rgb(VIOLET))
for i in range(len(X0)):
    for j in range(len(X0)):
        if W[i,j] > 0.075 and i != j:
            segments.append([X0[i], X0[j]])
            colors.append((*base_violet, 0.12 + 0.50*W[i,j]))
            widths.append(0.16 + 1.45*W[i,j])
ax.add_collection(LineCollection(segments, colors=colors, linewidths=widths, zorder=1))
step = layers[1] - layers[0]
ax.quiver(X0[:,0], X0[:,1], step[:,0], step[:,1], angles="xy", scale_units="xy", scale=1.0, color=GRAY, width=0.0062, alpha=0.60, zorder=2)
ax.scatter(X0[:,0], X0[:,1], s=DIRAC_MARKER_SIZE * 0.70, marker="o", color=RED, edgecolor="none", linewidth=0, zorder=3)
decorate(ax)
save_pdf(fig, out / "attention-edges.pdf", pad_inches=0.055)
plt.close(fig)

fig, ax = plt.subplots(figsize=(2.35, 2.05))
for i in range(len(X0)):
    pts = np.array([L[i] for L in layers])
    segs = np.stack([pts[:-1], pts[1:]], axis=1)
    cols = [(*interp_color(k/(len(layers)-2)), 0.50) for k in range(len(layers)-1)]
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=0.72, zorder=1))
for k, L in enumerate(layers):
    size = DIRAC_MARKER_SIZE * (0.36 if k not in [0, len(layers)-1] else 0.62)
    ax.scatter(L[:,0], L[:,1], s=size, marker="o", color=interp_color(k/(len(layers)-1)), edgecolor="none", linewidth=0, alpha=0.82, zorder=2+k)
decorate(ax)
save_pdf(fig, out / "depth-flow.pdf", pad_inches=0.055)
plt.close(fig)